# Guía de comandos y conceptos — Panadería, Grupo 4 (Power Rangers)

Este cuaderno recorre, en orden, lo que se hizo en el proyecto y los comandos que sirven como **evidencia** en la lista de cotejo (secciones A a G). Cada sección tiene tres partes: qué concepto se practica, el comando y qué se debe ver en pantalla.

**Cómo usarlo**

- Está pensado para **estudiar y ensayar** en Cloud Shell. En la defensa no se permite leer guiones ni ejecutar scripts preparados para las preguntas operativas: los comandos se escriben en vivo, así que conviene practicarlos hasta saberlos de memoria.
- Cada celda se ejecuta con `%%bash`. Las variables de la primera celda se comparten con todas las demás.
- Las celdas marcadas con ⚠️ **cambian o borran cosas**. Léelas antes de ejecutarlas.
- Para las capturas del PDF de evidencias: cada una debe mostrar el **reloj del sistema** y la salida de `gcloud config list` con el proyecto. La sección 1 ya hace las dos cosas.

| Sección | Qué se demuestra |
|---|---|
| A | Repositorio, README, diagrama, carnés, nombres del equipo |
| B | Microservicios, Dockerfiles y frontend |
| C | Imágenes en Artifact Registry |
| D | Terraform: plan, apply, nodos spot |
| E | Despliegue: Deployments, ConfigMap, Secret, PVC, IP pública, DNS |
| F | Operación: escalar, recrear pods, rolling update, logs |
| G | Costos y destrucción |

## 0. Preparación

Define las variables que usan todas las celdas. Cambia el proyecto solo si cambia la cuenta.

In [ ]:
%env PROYECTO=virtualizacion-proyecto-pr
%env REGION=us-central1
%env ZONA=us-central1-a
%env CLUSTER=panaderia-grupo-4
%env NS=panaderia-grupo-4
%env REPO=panaderia-repo

### 1. Identidad del proyecto y reloj (aparece en **todas** las capturas)

`gcloud config list` muestra la cuenta y el proyecto activos. El enunciado exige verlos junto con la hora del sistema.

In [ ]:
%%bash
date
gcloud config set project $PROYECTO >/dev/null 2>&1
gcloud config list

**Debe verse:** la fecha de hoy y `project = virtualizacion-proyecto-pr`.

Si el proyecto no coincide, ejecuta `gcloud config set project virtualizacion-proyecto-pr` antes de capturar.

## 2. Pre-vuelo (sección 8 del enunciado)

Antes de escribir código se confirma que la cuenta, la facturación, las APIs y Terraform están listos.

In [ ]:
%%bash
gcloud auth list
echo "--- facturación"
gcloud billing projects describe $PROYECTO --format="value(billingEnabled)"
echo "--- APIs necesarias"
gcloud services list --enabled --filter="config.name:(container.googleapis.com OR artifactregistry.googleapis.com OR compute.googleapis.com OR cloudresourcemanager.googleapis.com)" --format="value(config.name)"
echo "--- Terraform"
terraform version | head -1

**Debe verse:** `True` en facturación, las cuatro APIs y la versión de Terraform.

**Si falta una API:** `gcloud services enable container.googleapis.com artifactregistry.googleapis.com compute.googleapis.com cloudresourcemanager.googleapis.com`.

**Concepto:** Terraform necesita que las APIs ya estén habilitadas, porque lee datos del proyecto antes de crear nada. Fue el primer error que apareció (`accessNotConfigured`).

## 3. Sección A — Repositorio y trazabilidad

**Concepto:** el enunciado pide al menos 10 confirmaciones (commits) repartidas en 3 días distintos, sin alterar fechas.

In [ ]:
%%bash
cd ~/Virtualizacion_Proyecto_Final_PW
echo "Total de commits: $(git rev-list --count HEAD)"
echo "--- commits por día"
git log --format=%ad --date=short | sort | uniq -c
echo "--- últimos 5"
git log --oneline -5

**Debe verse:** más de 10 commits y al menos 3 fechas distintas.

### Identificación del equipo (A5) y carnés (A4)

El clúster y el namespace llevan el identificador del equipo, y los carnés están en la base de datos y en la interfaz.

In [ ]:
%%bash
kubectl get ns $NS
echo "--- carnés guardados en PostgreSQL"
kubectl exec -n $NS deploy/postgres -- sh -c 'PGPASSWORD=$POSTGRES_PASSWORD psql -U panaderia_admin -d productos_db -c "select nombre from productos where nombre like '"'"'%Carné%'"'"' order by nombre"'

**Debe verse:** el namespace `panaderia-grupo-4` y 5 filas con los carnés.

**Si no hay filas:** falta cargar los datos iniciales (sección 8, «Cargar los datos iniciales»).

## 4. Sección B — Los microservicios

**Concepto:** cada entidad tiene su propio servicio, su propia base de datos lógica y su propio Dockerfile. Pedidos referencia a Productos solo por el identificador, sin clave foránea, porque las bases están separadas.

In [ ]:
%%bash
cd ~/Virtualizacion_Proyecto_Final_PW
ls services/productos/Dockerfile services/pedidos/Dockerfile services/frontend/Dockerfile
echo "--- etapas del Dockerfile de Productos"
grep -n "^FROM\|^USER" services/productos/Dockerfile

**Debe verse:** tres Dockerfiles. En el de Productos aparecen dos etapas `FROM ... AS` y la línea `USER` que ejecuta el contenedor sin privilegios de administrador.

### CRUD completo de ambos servicios (B1 y B2)

Los comandos usan las IP públicas del clúster. La celda obtiene las direcciones por sí sola.

In [ ]:
%%bash
PROD=$(kubectl get svc productos -n $NS -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
PED=$(kubectl get svc pedidos -n $NS -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
echo "Productos: $PROD   Pedidos: $PED"
J='content-type: application/json'

echo "--- crear producto (POST)"
ID=$(curl -s -X POST http://$PROD:8000/productos -H "$J" -d '{"nombre":"Pan de demostración","precio":"4.50"}' | python3 -c "import sys,json;print(json.load(sys.stdin)['id'])")
echo $ID
echo "--- consultar (GET)";  curl -s http://$PROD:8000/productos/$ID; echo
echo "--- actualizar (PATCH)"; curl -s -X PATCH http://$PROD:8000/productos/$ID -H "$J" -d '{"precio":"5.00"}'; echo
echo "--- crear pedido (POST)"
PEDIDO=$(curl -s -X POST http://$PED:8000/pedidos -H "$J" -d "{\"producto_id\":\"$ID\",\"cantidad\":3}" | python3 -c "import sys,json;d=json.load(sys.stdin);print(d['id'])")
curl -s http://$PED:8000/pedidos/$PEDIDO; echo
echo "--- el total lo calcula el servidor: 3 x 5.00"

**Debe verse:** el producto con precio `5.00`, y un pedido con `precio_unitario` `5.00` y `total` `15.00`.

**Concepto:** el cliente solo envía `producto_id` y `cantidad`. El precio y el total los calcula el servidor; si el cliente envía `total`, la API responde `422`.

### Regla de integridad: no borrar un producto con pedidos

Esta regla se agregó al final del proyecto. Productos pregunta a Pedidos por HTTP antes de borrar.

In [ ]:
%%bash
PROD=$(kubectl get svc productos -n $NS -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
PED=$(kubectl get svc pedidos -n $NS -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
J='content-type: application/json'
ID=$(curl -s -X POST http://$PROD:8000/productos -H "$J" -d '{"nombre":"Pan con pedido","precio":"3.00"}' | python3 -c "import sys,json;print(json.load(sys.stdin)['id'])")
PEDIDO=$(curl -s -X POST http://$PED:8000/pedidos -H "$J" -d "{\"producto_id\":\"$ID\",\"cantidad\":2}" | python3 -c "import sys,json;print(json.load(sys.stdin)['id'])")

echo "1) borrar el producto teniendo un pedido -> debe responder 409"
curl -s -w "\nHTTP %{http_code}\n" -X DELETE http://$PROD:8000/productos/$ID
echo "2) el producto sigue existiendo -> 200"
curl -s -o /dev/null -w "HTTP %{http_code}\n" http://$PROD:8000/productos/$ID
echo "3) borrar el pedido y luego el producto -> 204 y 204"
curl -s -o /dev/null -w "HTTP %{http_code} " -X DELETE http://$PED:8000/pedidos/$PEDIDO
curl -s -o /dev/null -w "HTTP %{http_code}\n" -X DELETE http://$PROD:8000/productos/$ID

**Debe verse:** `409` con el código `product_has_orders`, luego `200`, y al final `204 204`.

**Pregunta típica:** *¿Por qué no basta con una clave foránea?* Porque cada servicio tiene su propia base de datos; la comprobación se hace por HTTP.

## 5. Sección C — Imágenes en Artifact Registry

**Concepto:** una imagen se identifica por la ruta completa `región-docker.pkg.dev/proyecto/repositorio/imagen:etiqueta`. Los manifiestos de Kubernetes solo pueden referenciar imágenes de ahí (C3).

### Construir y publicar (C1)

⚠️ Tarda varios minutos. Desde Cloud Shell, Docker Hub agota el tiempo de espera, por eso el script descarga las imágenes base del espejo de Google. Los envíos al registro pueden fallar de forma intermitente con `connection refused`; el script reintenta.

In [ ]:
%%bash
cd ~/Virtualizacion_Proyecto_Final_PW
bash scripts/publicar_imagenes.sh 2>&1 | tail -15

### Listar el repositorio (C2)

In [ ]:
%%bash
REG=$REGION-docker.pkg.dev/$PROYECTO/$REPO
gcloud artifacts docker images list $REG --include-tags --format="table(package.basename(), tags, createTime)"

**Debe verse:** `frontend`, `pedidos`, `productos` con la etiqueta `1.1.0`, `productos` también con `1.1.1`, y `postgres` con `16`.

### Las imágenes que usan los manifiestos (C3)

In [ ]:
%%bash
cd ~/Virtualizacion_Proyecto_Final_PW
grep -h "image:" deploy/k8s/*.yaml | sort -u

**Debe verse:** todas las líneas empiezan con `us-central1-docker.pkg.dev/virtualizacion-proyecto-pr/panaderia-repo/`.

## 6. Sección D — Infraestructura como código con Terraform

**Concepto:** Terraform describe la infraestructura en archivos `.tf` y la crea de forma repetible. No gestiona los objetos de Kubernetes, que se aplican con `kubectl`.

| Archivo | Qué declara |
|---|---|
| `provider.tf` | El proveedor de Google Cloud |
| `variables.tf` | `project_id`, `region`, `zone`, `cluster_name`, sin valores sensibles |
| `vpc.tf` | Red y subred con rangos para pods y Services |
| `gke.tf` | Clúster GKE zonal y node pool de 2 nodos `e2-small` spot |
| `artifact_registry.tf` | Repositorio Docker y permiso de lectura para los nodos |
| `outputs.tf` | Nombre del clúster, comando de conexión y ruta del registro |

In [ ]:
%%bash
cd ~/Virtualizacion_Proyecto_Final_PW/infra/terraform
ls *.tf terraform.tfvars.example
echo "--- variables (D2)"
grep -E "^variable|default" variables.tf
echo "--- valores usados"
cat terraform.tfvars

**Debe verse:** cuatro variables sin contraseñas ni claves. `terraform.tfvars` contiene solo el ID del proyecto, la región, la zona y el nombre del clúster.

### Inicializar y planificar (D3)

`init` descarga el proveedor. `plan` muestra qué se va a crear **sin crear nada**. La captura del plan debe hacerse **antes** del `apply`.

In [ ]:
%%bash
cd ~/Virtualizacion_Proyecto_Final_PW/infra/terraform
terraform init -no-color | tail -4
echo "=================== PLAN"
terraform plan -no-color -out=tfplan

**Debe verse:** al final `Plan: 6 to add, 0 to change, 0 to destroy.` Los seis recursos son el repositorio, el permiso de lectura, la red, la subred, el clúster y el node pool.

Si el clúster ya existe, el plan dirá `No changes`. Eso también es correcto, pero no sirve como evidencia de creación.

### Aplicar (D4)

⚠️ **Crea recursos que cuestan dinero.** El clúster tarda unos 8 minutos.

In [ ]:
%%bash
cd ~/Virtualizacion_Proyecto_Final_PW/infra/terraform
date
terraform apply -no-color tfplan | tail -20

**Debe verse:** `Apply complete! Resources: 6 added, 0 changed, 0 destroyed.` y las tres salidas (`artifact_registry_url`, `cluster_name`, `gcloud_connect_command`).

### Nodos spot (D5)

Los nodos spot son máquinas más baratas que Google puede reclamar. El enunciado exige exactamente 2 nodos `e2-small`.

In [ ]:
%%bash
gcloud container node-pools list --cluster $CLUSTER --zone $ZONA --format="table(name, config.machineType, config.spot, initialNodeCount)"
echo "--- nodos"
kubectl get nodes -o wide

**Debe verse:** `e2-small` con `spot: True` y 2 nodos en estado `Ready`.

Si `kubectl` dice que no hay contexto, conecta primero con la celda siguiente.

## 7. Conectar `kubectl` al clúster

`get-credentials` escribe en `~/.kube/config` las credenciales del clúster.

In [ ]:
%%bash
gcloud container clusters get-credentials $CLUSTER --zone $ZONA --project $PROYECTO
kubectl config current-context

## 8. Sección E — Despliegue en GKE

**Concepto:** los objetos de Kubernetes están en `deploy/k8s/` y se aplican **en orden**: namespace, ConfigMap, Secret, volumen y base de datos, migraciones y, por último, los servicios.

### Crear el Secret

El Secret guarda las contraseñas. El script las genera al azar y **no las imprime**. Si ya existe, no lo cambia.

In [ ]:
%%bash
cd ~/Virtualizacion_Proyecto_Final_PW
bash scripts/crear_secret_gke.sh

### Desplegar

⚠️ Aplica los manifiestos y espera a que cada pieza esté lista.

In [ ]:
%%bash
cd ~/Virtualizacion_Proyecto_Final_PW
bash deploy/k8s/apply.sh 2>&1 | tail -25

### Estado de todos los recursos (E1)

In [ ]:
%%bash
date
kubectl get all,pvc -n $NS

**Debe verse:** los pods de `frontend`, `pedidos`, `productos` y `postgres` en `Running`, los dos Jobs de migración en `Completed`, tres Services de tipo `LoadBalancer` con IP externa, y el PVC `postgres-pvc` en `Bound`.

**Si una IP externa dice `<pending>`:** espera entre 1 y 3 minutos. Si persiste, revisa la cuota de direcciones IP de la región.

### Cargar los datos iniciales y los carnés (A4)

El script es idempotente: si los productos ya existen, no los duplica.

In [ ]:
%%bash
cd ~/Virtualizacion_Proyecto_Final_PW
PROD=$(kubectl get svc productos -n $NS -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
PED=$(kubectl get svc pedidos -n $NS -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
PRODUCTOS_URL=http://$PROD:8000 PEDIDOS_URL=http://$PED:8000 python3 scripts/seed_database.py 2>&1 | tail -8

### ConfigMap: configuración no sensible (E2)

**Concepto:** un ConfigMap guarda valores que no son secretos (URLs, tiempos). Los pods los reciben como variables de entorno, así el mismo contenedor funciona en cualquier entorno.

In [ ]:
%%bash
echo "--- declarado"
kubectl get configmap panaderia-config -n $NS -o yaml | sed -n '1,25p'
echo "--- consumido dentro de un pod"
kubectl exec -n $NS deploy/pedidos -- sh -c 'echo PRODUCTOS_BASE_URL=$PRODUCTOS_BASE_URL; echo PRODUCTOS_TIMEOUT_SECONDS=$PRODUCTOS_TIMEOUT_SECONDS'

**Debe verse:** el ConfigMap con sus claves, y las mismas variables dentro del pod de Pedidos.

### Secret: credenciales (E3)

**Concepto:** un Secret guarda credenciales. Se almacenan en base64, que **no es cifrado**; sirve para no escribirlas en los manifiestos ni en Git.

In [ ]:
%%bash
echo "--- claves del Secret (sin valores)"
kubectl describe secret panaderia-secrets -n $NS | sed -n '1,20p'
echo "--- consumido dentro del pod de Productos (contraseña oculta)"
kubectl exec -n $NS deploy/productos -- sh -c 'echo $DATABASE_URL' | sed -E 's#(://[^:]+:)[^@]+@#\1********@#'

**Debe verse:** cinco claves con su tamaño en bytes, y la cadena de conexión con la contraseña como `********`.

**Importante:** en la captura **no** muestres la contraseña real.

### Volumen persistente: los datos sobreviven al pod (E4)

**Concepto:** un pod es desechable; su disco interno desaparece con él. El PersistentVolumeClaim reserva un disco aparte, montado en la ruta de datos de PostgreSQL, que sobrevive.

La prueba: contar productos, **borrar el pod de PostgreSQL**, esperar a que Kubernetes cree otro y contar de nuevo.

⚠️ Interrumpe la aplicación unos segundos.

In [ ]:
%%bash
PROD=$(kubectl get svc productos -n $NS -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
cuenta() { curl -s "http://$PROD:8000/productos?limit=1" | python3 -c "import sys,json;print(json.load(sys.stdin)['total'])"; }

echo "PVC:"; kubectl get pvc -n $NS
echo "Productos antes:  $(cuenta)"
echo "--- borrando el pod de PostgreSQL"
kubectl delete pod -n $NS -l app.kubernetes.io/name=postgres
kubectl wait -n $NS --for=condition=ready pod -l app.kubernetes.io/name=postgres --timeout=240s
sleep 15
echo "Productos después: $(cuenta)"
kubectl get pods -n $NS -l app.kubernetes.io/name=postgres

**Debe verse:** el mismo número antes y después, y un pod nuevo con otro nombre y pocos segundos de edad.

**Si los datos se pierden:** el volumen no está montado en la ruta de datos del motor; revisa el `mountPath` en `04-postgres.yaml`.

### IP pública (E5)

**Concepto:** un Service de tipo `LoadBalancer` pide a Google una dirección IP pública que reparte el tráfico entre los pods.

In [ ]:
%%bash
kubectl get svc -n $NS -o custom-columns=NOMBRE:.metadata.name,TIPO:.spec.type,IP_PUBLICA:.status.loadBalancer.ingress[0].ip,PUERTOS:.spec.ports[*].port
FRONT=$(kubectl get svc frontend -n $NS -o jsonpath='{.status.loadBalancer.ingress[0].ip}')
echo ""
echo "Abre en el navegador (HTTP, no HTTPS):   http://$FRONT"
curl -s -o /dev/null -w "Código de respuesta: %{http_code}\n" http://$FRONT/

**Debe verse:** el frontend con IP externa y código `200`.

**Importante:** la dirección es `http://`, no `https://`. Algunos navegadores, como Brave, la convierten en HTTPS y aparece `ERR_CONNECTION_TIMED_OUT`.

La captura de E5.2 es la página en el navegador **externo al clúster**, con la barra de direcciones visible.

### Comunicación interna por nombre DNS (E6)

**Concepto:** los pods cambian de IP cada vez que se recrean. Por eso se usa el nombre del Service (`productos`, `pedidos`), que Kubernetes resuelve por DNS interno.

In [ ]:
%%bash
echo "--- el frontend llega a Productos y Pedidos por nombre"
kubectl exec -n $NS deploy/frontend -- wget -q -O - http://productos:8000/health/live; echo
kubectl exec -n $NS deploy/frontend -- wget -q -O - http://pedidos:8000/health/live; echo
echo "--- el proxy de Nginx apunta a los nombres, no a IP"
kubectl exec -n $NS deploy/frontend -- grep proxy_pass /etc/nginx/nginx.conf
echo "--- Pedidos llama a Productos con el nombre completo del Service"
kubectl exec -n $NS deploy/pedidos -- sh -c 'echo $PRODUCTOS_BASE_URL'

**Debe verse:** `{"status":"ok"}` dos veces, líneas `proxy_pass http://productos:8000/...` y la URL con `.svc.cluster.local`.

## 9. Sección F — Operación de Kubernetes

### Escalar a 3 réplicas de forma declarativa (F1)

**Concepto:** "declarativo" significa que se **cambia el archivo** y se aplica, en vez de dar una orden puntual. El estado deseado queda escrito en el YAML.

⚠️ Cambia el archivo `06-productos.yaml` en esta copia local. Al final se puede revertir con `git checkout deploy/k8s/06-productos.yaml`.

In [ ]:
%%bash
cd ~/Virtualizacion_Proyecto_Final_PW
echo "--- antes"; grep -n "replicas" deploy/k8s/06-productos.yaml
sed -i 's/replicas: 2/replicas: 3/' deploy/k8s/06-productos.yaml
echo "--- después"; grep -n "replicas" deploy/k8s/06-productos.yaml
kubectl apply -f deploy/k8s/06-productos.yaml
kubectl rollout status deployment/productos -n $NS --timeout=240s
echo "--- réplicas en ejecución"
kubectl get pods -n $NS -l app.kubernetes.io/name=productos
kubectl get deployment productos -n $NS

**Debe verse:** `replicas: 3` en el archivo, `deployment.apps/productos configured` y tres pods `Running`.

Para volver a 2: `sed -i 's/replicas: 3/replicas: 2/' deploy/k8s/06-productos.yaml` y `kubectl apply -f deploy/k8s/06-productos.yaml`.

### Eliminar un pod y verlo recrear (F2)

**Concepto:** el Deployment vigila que haya el número de réplicas declarado. Si un pod desaparece, crea otro. A esto se le llama **autorrecuperación**.

⚠️ Borra un pod de Productos (los otros siguen atendiendo).

In [ ]:
%%bash
POD=$(kubectl get pods -n $NS -l app.kubernetes.io/name=productos -o jsonpath='{.items[0].metadata.name}')
echo "Pod que se va a borrar: $POD"
kubectl get pods -n $NS -l app.kubernetes.io/name=productos
echo "--- borrando"
kubectl delete pod $POD -n $NS
for i in 1 2 3 4 5 6; do
  sleep 4
  echo "--- t=$((i*4))s"; kubectl get pods -n $NS -l app.kubernetes.io/name=productos --no-headers
done

**Debe verse:** el pod borrado desaparece y aparece uno nuevo (nombre distinto, edad de segundos) hasta volver a tener todos los pods en `1/1 Running`.

### Actualización de versión con rolling update (F3)

**Concepto:** un *rolling update* cambia la versión poco a poco: crea un pod nuevo, espera a que esté listo y recién entonces retira uno viejo. La estrategia es `maxSurge: 1` y `maxUnavailable: 0`, es decir, nunca baja del número deseado.

⚠️ Cambia la imagen de Productos de `1.1.0` a `1.1.1`. La etiqueta `1.1.1` debe estar publicada en el registro (sección 5).

In [ ]:
%%bash
REG=$REGION-docker.pkg.dev/$PROYECTO/$REPO
echo "--- estrategia"
kubectl get deployment productos -n $NS -o jsonpath='{.spec.strategy}{"\n"}'
echo "--- versión actual"
kubectl get deployment productos -n $NS -o jsonpath='{.spec.template.spec.containers[0].image}{"\n"}'

kubectl set image deployment/productos productos=$REG/productos:1.1.1 -n $NS
for i in 1 2 3 4 5; do
  sleep 6
  echo "--- t=$((i*6))s"; kubectl get pods -n $NS -l app.kubernetes.io/name=productos --no-headers
done
kubectl rollout status deployment/productos -n $NS --timeout=240s
echo "--- versión nueva"
kubectl get deployment productos -n $NS -o jsonpath='{.spec.template.spec.containers[0].image}{"\n"}'
echo "--- historial"
kubectl rollout history deployment/productos -n $NS

**Debe verse:** durante la actualización conviven pods viejos y nuevos, al final `successfully rolled out` y la imagen `productos:1.1.1`.

**Para volver atrás:** `kubectl rollout undo deployment/productos -n $NS`.

**Nota:** en una prueba, durante el reinicio algunas solicitudes externas no recibieron respuesta, aunque la estrategia no permite quedar por debajo del mínimo. Es un tema que conviene conocer por si lo preguntan (ver `docs/cambios-pendientes.md`, P1).

### Registros de un pod (F4)

**Concepto:** cada contenedor escribe su salida estándar y Kubernetes la guarda. `kubectl logs` la lee sin entrar al pod.

In [ ]:
%%bash
kubectl logs deployment/productos -n $NS --tail=15
echo "=================== Pedidos"
kubectl logs deployment/pedidos -n $NS --tail=8

**Debe verse:** líneas de acceso de FastAPI (`GET /health/ready ... 200 OK`) con la fecha.

Útiles: `-f` para seguir en vivo, `--previous` para ver el contenedor anterior si se reinició, y `-c` para elegir contenedor.

## 10. Diagnóstico de fallas (pregunta individual)

La defensa incluye una pregunta de diagnóstico. El método es siempre el mismo: **ver el estado, describir el objeto, leer los logs**.

```
kubectl get pods -n $NS                 # ¿en qué estado está?
kubectl describe pod <nombre> -n $NS    # ¿qué eventos tiene?
kubectl logs <nombre> -n $NS            # ¿qué dice la aplicación?
```

| Síntoma | Causa probable | Cómo comprobarlo |
|---|---|---|
| `ImagePullBackOff` / `ErrImagePull` | Etiqueta inexistente, ruta equivocada o nodos sin permiso de lectura al registro (error `403`) | `kubectl describe pod` y leer el evento `Failed`. Revisar el rol `artifactregistry.reader` de la cuenta de servicio de los nodos |
| `CrashLoopBackOff` | La aplicación falla al arrancar: variable faltante o base de datos inaccesible | `kubectl logs <pod> --previous` |
| PVC en `Pending` | No hay StorageClass por defecto | `kubectl get storageclass` |
| Se pierden los datos al recrear PostgreSQL | El volumen no está montado en la ruta de datos | Revisar `mountPath` y `subPath` en el Deployment |
| `EXTERNAL-IP` en `<pending>` | El balanceador tarda 1 a 3 minutos, o falta cuota de IP | `kubectl describe svc <nombre>` y revisar la cuota |
| El frontend no alcanza los servicios | Se usó una IP de pod en vez del nombre del Service | Usar `http://productos:8000` |
| Migración que no termina | Contraseña distinta en el Secret o la base no existe | `kubectl logs job/productos-migration-job -n $NS` |
| `terraform apply` falla por una API | La API no está habilitada | `gcloud services enable ...` y repetir |
| `terraform destroy` deja recursos vivos | Se crearon a mano con `kubectl` o desde la consola, fuera del estado | Eliminarlos aparte (celda siguiente) |
| Página no carga en el navegador | El navegador convirtió la dirección a `https://` | Escribir `http://` y desactivar la actualización a HTTPS |
| `409` al borrar un producto | Tiene pedidos asociados | Borrar los pedidos, o desactivar el producto con `activo=false` |

Comandos para investigar rápido:

In [ ]:
%%bash
echo "--- eventos recientes del namespace"
kubectl get events -n $NS --sort-by=.lastTimestamp | tail -12
echo "--- pods que no están Running"
kubectl get pods -n $NS --field-selector=status.phase!=Running
echo "--- consumo"
kubectl top pods -n $NS 2>/dev/null || echo "(métricas no disponibles todavía)"

## 11. Preguntas conceptuales — respuestas cortas

- **¿Qué es un Pod?** La unidad más pequeña de Kubernetes: uno o más contenedores que comparten red y almacenamiento. Es desechable.
- **¿Qué es un Deployment?** El objeto que declara cuántas réplicas de un pod debe haber y cómo actualizarlas. Si un pod muere, crea otro.
- **¿Qué es un Service?** Una dirección estable (nombre DNS e IP) delante de un grupo de pods. `ClusterIP` es interna; `LoadBalancer` agrega una IP pública.
- **¿ConfigMap o Secret?** El ConfigMap guarda configuración no sensible; el Secret guarda credenciales. Ninguno cifra por sí solo: base64 no es cifrado.
- **¿Para qué sirve el PVC?** Reserva un disco independiente del pod, para que los datos de PostgreSQL sobrevivan a su recreación.
- **¿Qué es un namespace?** Una división lógica del clúster. Aquí aísla todo en `panaderia-grupo-4`.
- **¿Qué hace Terraform y qué no?** Crea la infraestructura (red, clúster, registro). No gestiona los objetos de Kubernetes, que se aplican con `kubectl`.
- **¿Para qué sirve el estado de Terraform?** Es el registro de lo que Terraform creó. Sin él no puede saber qué borrar en `destroy`.
- **¿Qué es un nodo spot?** Una máquina con descuento que Google puede reclamar. Sirve para ahorrar en un proyecto académico.
- **¿Por qué dos bases separadas?** Cada microservicio es dueño de sus datos. Pedidos guarda una copia histórica del nombre y precio, así que no depende de que Productos esté disponible para consultarse.
- **¿Qué pasa si Productos se cae?** Crear un pedido responde `503`; consultar pedidos existentes sigue funcionando.
- **¿Qué es un rolling update?** Reemplazar los pods de a uno, esperando que el nuevo esté listo, para no interrumpir el servicio.
- **¿Para qué sirven las sondas `live` y `ready`?** `live` indica si el proceso está vivo; `ready` si puede recibir tráfico (por ejemplo, si la base responde).
- **¿Por qué los Jobs de migración van aparte?** Para que dos réplicas no migren la base a la vez.

## 12. Sección G — Costos y cierre responsable

### Estimación (G1)

Debe existir **antes** del despliegue. Escribirla en `docs/estimacion-costos.md` con los precios que muestra la consola de Google Cloud (calculadora de precios) para la región `us-central1`:

- 2 nodos `e2-small` spot
- disco persistente de 10 GB y discos de los nodos
- Artifact Registry (almacenamiento)
- 3 balanceadores de carga (frontend, productos, pedidos)
- tarifa de administración de GKE

### Destruir la infraestructura (G2)

⚠️ **Borra el clúster y todo lo que creó Terraform.** Hazlo **después de la defensa**. Antes, borra lo que se creó a mano con `kubectl`, para que no queden balanceadores ni discos vivos fuera del estado de Terraform.

In [ ]:
%%bash
echo "ANTES de destruir: el namespace crea balanceadores y discos fuera de Terraform"
kubectl delete namespace $NS --timeout=240s

In [ ]:
%%bash
cd ~/Virtualizacion_Proyecto_Final_PW/infra/terraform
date
terraform destroy -no-color -auto-approve | tail -15

**Debe verse:** `Destroy complete! Resources: 6 destroyed.`

### Comprobar que no quedó nada vivo

In [ ]:
%%bash
echo "clústeres:";            gcloud container clusters list --format="value(name)"
echo "balanceadores:";        gcloud compute forwarding-rules list --format="value(name)"
echo "discos:";               gcloud compute disks list --format="value(name)"
echo "repositorios:";         gcloud artifacts repositories list --format="value(name)"
echo "redes (solo 'default' es normal):"; gcloud compute networks list --format="value(name)"

**Debe verse:** todas las listas vacías, salvo la red `default`.

### Panel de facturación (G3)

Se captura en la consola web: **Facturación → Informes**, con el rango de fechas del proyecto y agrupado por producto. Debe mostrarse el gasto final. Fecha límite de la evidencia de destrucción: **viernes 13 de noviembre, 23:59**.

## Resumen del orden en una defensa

1. `date` y `gcloud config list` (siempre primero).
2. `kubectl get all,pvc -n panaderia-grupo-4` (E1).
3. La aplicación en el navegador con `http://IP` (E5).
4. ConfigMap y Secret dentro de un pod (E2, E3).
5. Borrar el pod de PostgreSQL y comprobar que los datos siguen (E4).
6. Escalar a 3 réplicas editando el YAML (F1).
7. Borrar un pod de Productos y ver que se recrea (F2).
8. Rolling update `1.1.0` a `1.1.1` (F3).
9. Logs de un pod (F4).

Practica cada paso hasta poder escribirlo sin mirar este cuaderno.